# 00. ETL

Запуск ETL из ноутбука для `HI-Small_Trans.csv`. То же самое делает команда `aml-etl`.

Первый день идёт только в историю счетов, хвост в конце симуляции выкидывается. Каждый запуск пишется в новую папку, а `artifacts/runs/active.json` обновляется только если всё записалось до конца.

In [ ]:
import json
import sys
from pathlib import Path
from time import perf_counter

import pandas as pd
import pyarrow.parquet as pq

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from aml.cli.etl import run_etl

In [ ]:
SOURCE = PROJECT_ROOT / "data" / "raw" / "HI-Small_Trans.csv"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "runs"
SMOKE = False  # True = быстрый прогон только на каждом 50-м счёте

print(SOURCE, f"{SOURCE.stat().st_size / 1024**2:.0f} МБ")

## Запуск

Данные режутся по дням на warmup → train → val → threshold → test. Исторические признаки видят только то, что было раньше по времени.

In [ ]:
start = perf_counter()
RUN_DIR = run_etl(SOURCE, OUTPUT_DIR, smoke=SMOKE)
print("готово:", RUN_DIR)
print(f"заняло {(perf_counter() - start) / 60:.1f} мин")

## Проверка

Смотрю, что всё сохранилось и количество строк сходится. Строки считаю по метаданным parquet, чтобы не грузить все данные заново.

In [4]:
manifest = json.loads((RUN_DIR / "data_manifest.json").read_text())
configuration = json.loads((RUN_DIR / "configuration.json").read_text())
assert (RUN_DIR / "SUCCESS").exists()
assert Path(manifest["source"]["path"]) == SOURCE.resolve()
assert configuration["smoke"] == SMOKE

rows = {}
for name in ["train", "val", "threshold", "test"]:
    n = pq.ParquetFile(RUN_DIR / "data" / f"{name}.parquet").metadata.num_rows
    n_time = pq.ParquetFile(RUN_DIR / "data" / f"{name}_time.parquet").metadata.num_rows
    assert n == n_time > 0, name
    rows[name] = n
print("строк:", rows)

# сохранённые + выкинутые строки должны дать весь исходный файл
excluded = manifest["transformation"]["excluded_rows"]
print("выкинуто:", excluded)
assert sum(rows.values()) + sum(excluded.values()) == manifest["source"]["extracted_rows"]

display(pd.DataFrame(manifest["transformation"]["periods"]).T)
display(pd.Series(json.loads((RUN_DIR / "quality_report.json").read_text())))

строк: {'train': 2134000, 'val': 482751, 'threshold': 482773, 'test': 862792}
выкинуто: {'warmup': 1114921, 'tail': 1108}


,rows,min_timestamp,max_timestamp,positives
warmup,1114921,2022-09-01T00:00:00,2022-09-01T23:59:00,322
train,2134000,2022-09-02T00:00:00,2022-09-06T23:59:00,2208
val,482751,2022-09-07T00:00:00,2022-09-07T23:59:00,497
threshold,482773,2022-09-08T00:00:00,2022-09-08T23:59:00,539
test,862792,2022-09-09T00:00:00,2022-09-10T23:59:00,956
tail,1108,2022-09-11T00:05:00,2022-09-18T16:18:00,655


rows                                                         5078345
duplicate_rows                                                     9
zero_amounts                                                       0
self_transfers                                                591212
negatives                                                    5073168
positives                                                       5177
positive_rate                                               0.001019
min_timestamp                                    2022-09-01T00:00:00
max_timestamp                                    2022-09-18T16:18:00
payment_formats    {'ACH': 600797, 'Bitcoin': 146091, 'Cash': 490...
currencies         {'Australian Dollar': 275280, 'Bitcoin': 29421...
daily              [{'day': '2022-09-01', 'rows': 1114921, 'posit...
dtype: object